In [2]:
import csv
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

In [3]:
df = pd.read_csv(r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\data analysis\data\Constitutional Alignment_January 8, 2026_07.42.csv')

In [4]:
# Split the data into constitution and principle conditions

# Add a bool feature to filter out only Prolific submissions
df['PID_Correct_Len'] = df['PROLIFIC_PID'].str.len() ==  24
dfs = {group_name: group for group_name, group in df.groupby('Version')}

const_df = dfs['Constitution']
princ_df = dfs['Principle']

# pare down the irrelevant fields

const_fields = ['con_b_personal','con_b_pers_elab','con_b_govern','con_b_govern_elab','con_b_matrix_1',
                'con_b_matrix_2','con_b_matrix_3','con_a_personal','con_a_pers_elab','con_a_gov','con_a_gov_elab',
                'con_a_matrix_1','con_a_matrix_2','con_a_matrix_3', 'partDemo.age',	'partDemo.gender',
             	'partDemo.gender_7_TEXT', 'partDemo.ethnicity',	'partDemo.ethnicity_7_TEXT', 'partDemo.education',
                'PROLIFIC_PID', 'Const_Order', 'PID_Correct_Len'
]


const_df = const_df[const_fields]

princ_fields = ['partDemo.age',	'partDemo.gender', 'partDemo.gender_7_TEXT', 'partDemo.ethnicity', 'partDemo.ethnicity_7_TEXT', 
                'partDemo.education', 'PROLIFIC_PID', '2_Principle Q1_1','2_Principle Q1_2','2_Principle Q2_1','2_Principle Q2_2',
                '2_Principle Q2_3','2_Principle Q2_4','2_Principle Q2_5','2_Principle Q2_6','2_Principle Q2_7','3_Principle Q1_1',
                '3_Principle Q1_2','3_Principle Q2_1','3_Principle Q2_2','3_Principle Q2_3','3_Principle Q2_4','3_Principle Q2_5',
                '3_Principle Q2_6','3_Principle Q2_7','4_Principle Q1_1','4_Principle Q1_2','4_Principle Q2_1','4_Principle Q2_2',
                '4_Principle Q2_3','4_Principle Q2_4','4_Principle Q2_5','4_Principle Q2_6','4_Principle Q2_7','5_Principle Q1_1',
                '5_Principle Q1_2','5_Principle Q2_1','5_Principle Q2_2','5_Principle Q2_3','5_Principle Q2_4','5_Principle Q2_5',
                '5_Principle Q2_6','5_Principle Q2_7','6_Principle Q1_1','6_Principle Q1_2','6_Principle Q2_1','6_Principle Q2_2',
                '6_Principle Q2_3','6_Principle Q2_4','6_Principle Q2_5','6_Principle Q2_6','6_Principle Q2_7','7_Principle Q1_1',
                '7_Principle Q1_2','7_Principle Q2_1','7_Principle Q2_2','7_Principle Q2_3','7_Principle Q2_4','7_Principle Q2_5',
                '7_Principle Q2_6','7_Principle Q2_7','attention_check_1_1','attention_check_1_2','attention_check_2_1',
                'attention_check_2_2','attention_check_2_3','attention_check_2_4','attention_check_2_5','attention_check_2_6',
                'attention_check_2_7','PID_Correct_Len', 'RESP_P1', 'RESP_P2','RESP_P3','RESP_P4','RESP_P5','RESP_P6'
]
princ_df = princ_df[princ_fields]

princ_df_filter = princ_df[princ_df['PID_Correct_Len'] == True]



In [5]:
princ_df_filter.head(10)

,partDemo.age,partDemo.gender,partDemo.gender_7_TEXT,partDemo.ethnicity,partDemo.ethnicity_7_TEXT,partDemo.education,PROLIFIC_PID,2_Principle Q1_1,2_Principle Q1_2,2_Principle Q2_1,...,attention_check_2_5,attention_check_2_6,attention_check_2_7,PID_Correct_Len,RESP_P1,RESP_P2,RESP_P3,RESP_P4,RESP_P5,RESP_P6
5,10,4,NaN,6,NaN,7,615e4e76151c3bb90b6bd868,5,5,5,...,1,1,1,True,P11,P15,P08,P06,P05,P17
7,9,4,NaN,6,NaN,10,67203bff74dea717e8b9e1ea,2,2,2,...,1,1,1,True,P12,P03,P05,P15,P01,P16
8,12,5,NaN,6,NaN,13,62cd9b920ed1b7c1648818d3,4,4,5,...,1,1,1,True,P12,P01,P11,P15,P05,P08
9,8,4,NaN,2,NaN,10,665b7b047373d8da553237a6,5,5,5,...,1,1,1,True,P10,P11,P09,P04,P19,P18
11,8,5,NaN,6,NaN,9,61008b27700cd8ceddeb9ebb,4,4,3,...,1,1,1,True,P18,P02,P09,P06,P12,P14
12,11,4,NaN,6,NaN,5,5bcbaa10b9ce3200017738ee,4,5,5,...,1,1,1,True,P20,P17,P08,P11,P03,P01
15,10,4,NaN,6,NaN,8,662190146ef67bab626e48cb,5,5,5,...,1,1,1,True,P17,P15,P16,P02,P05,P03
16,11,5,NaN,"3,6",NaN,10,647dde92d730823a9e5c7f0f,3,3,3,...,1,1,1,True,P12,P16,P15,P08,P04,P09
18,8,5,NaN,6,NaN,11,60fcdf04d033481fde0e2298,5,5,5,...,1,1,1,True,P07,P17,P09,P18,P08,P11
19,10,4,NaN,6,NaN,7,633dbb46887d82536a9ee265,5,5,4,...,1,1,1,True,P02,P17,P10,P13,P08,P14


In [6]:
# List of response columns and their corresponding question set prefixes
resp_cols = [f'RESP_P{i}' for i in range(1, 7)]
qset_prefixes = [f'{i+1}_Principle' for i in range(1, 7)]
q_cols = [f'Q1_1', 'Q1_2'] + [f'Q2_{i}' for i in range(1, 8)]

# All other fields to retain (excluding RESP_P1-6 and question set columns)
other_fields = [col for col in princ_df_filter.columns if not (
    col.startswith('RESP_P') or any(col.startswith(f'{i}_Principle') for i in range(2, 8))
)]

rows = []

for idx, row in princ_df_filter.iterrows(): # for each row
    for resp_idx, resp_col in enumerate(resp_cols): # for each RESP_P{i} column
        resp_val = row[resp_col] # respect_value = value in RESP_P{i}
        if pd.notnull(resp_val): # if the response value is not null
            prefix = qset_prefixes[resp_idx] # get the corresponding question-set prefix
            q_data = {} # question data
            for q in q_cols: # for each question in q cols 
                colname = f'{prefix} {q}' #col name is the combination of prefix + string q in q_cols
                if colname in princ_df_filter.columns: # if that col name exists in the dataframe
                    q_data[q] = pd.to_numeric(row[colname], errors='coerce') # add to q_data dict
                else:
                    q_data[q] = None
            # Collect all other fields
            base_data = row[other_fields].to_dict() #gather metadata
            # Add response set and question answers
            base_data['response_set'] = resp_val # add response value to base data
            base_data.update(q_data) # add q_data to base data
            rows.append(base_data) # append the combined data to rows

# Create the new DataFrame
transformed_df = pd.DataFrame(rows)

#### Should we add in a filter for attention check? Some are coming in empty. 

In [7]:
# VISUAL: Distribution of answers by response_set (with totals)


question_cols = ['Q1_1', 'Q1_2'] + [f'Q2_{i}' for i in range(1, 8)]

for q in question_cols:
    print(f"Distribution for {q} by response_set:")

    table = (
        transformed_df
            .groupby('response_set')[q]
            .value_counts()
            .unstack(fill_value=0)
    )

    # Add horizontal totals (row totals)
    table['Total'] = table.sum(axis=1)

    # Add vertical totals (column totals + grand total)
    table.loc['Total'] = table.sum(axis=0)

    display(table)

    print("\n" + "-" * 100 + "\n")


Distribution for Q1_1 by response_set:


Q1_1,1.0,2.0,3.0,4.0,5.0,6.0,Total
response_set,,,,,,,
P01,1,0,2,3,9,1,16
P02,0,1,1,2,9,0,13
P03,0,0,0,2,10,0,12
P04,0,0,1,4,10,0,15
P05,0,1,0,4,9,0,14
P06,0,0,2,0,9,0,11
P07,0,1,0,0,8,0,9
P08,0,0,0,2,7,1,10
P09,0,1,2,3,6,0,12



----------------------------------------------------------------------------------------------------

Distribution for Q1_2 by response_set:


Q1_2,1.0,2.0,3.0,4.0,5.0,6.0,Total
response_set,,,,,,,
P01,1,0,1,3,10,1,16
P02,1,0,1,3,8,0,13
P03,0,0,1,3,8,0,12
P04,0,0,3,2,10,0,15
P05,0,2,0,7,5,0,14
P06,0,1,1,1,8,0,11
P07,0,1,0,0,8,0,9
P08,0,0,0,3,6,1,10
P09,0,1,2,4,5,0,12



----------------------------------------------------------------------------------------------------

Distribution for Q2_1 by response_set:


...


----------------------------------------------------------------------------------------------------

Distribution for Q2_6 by response_set:


Q2_6,1.0,2.0,3.0,4.0,5.0,6.0,Total
response_set,,,,,,,
P01,0,1,2,5,8,0,16
P02,0,0,0,6,7,0,13
P03,0,0,0,3,8,1,12
P04,0,0,2,6,7,0,15
P05,1,0,1,6,6,0,14
P06,0,1,0,4,6,0,11
P07,0,0,0,1,8,0,9
P08,0,0,2,3,4,1,10
P09,0,1,3,5,3,0,12



----------------------------------------------------------------------------------------------------

Distribution for Q2_7 by response_set:


Q2_7,1.0,2.0,3.0,4.0,5.0,6.0,Total
response_set,,,,,,,
P01,1,1,2,3,8,1,16
P02,1,0,0,6,5,1,13
P03,0,0,2,4,6,0,12
P04,0,1,4,4,5,1,15
P05,1,2,1,6,4,0,14
P06,0,0,2,1,8,0,11
P07,0,0,0,2,6,1,9
P08,0,3,2,0,4,1,10
P09,1,1,3,3,4,0,12



----------------------------------------------------------------------------------------------------



In [8]:
# Tabulated data

question_cols = ['Q1_1', 'Q1_2'] + [f'Q2_{i}' for i in range(1, 8)]

df_counts = (
    transformed_df
        .melt(
            id_vars='response_set',
            value_vars=question_cols,
            var_name='question',
            value_name='value'
        )
        .groupby(['response_set', 'question', 'value'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
)

for i in range(1, 7):
    if i not in df_counts.columns:
        df_counts[i] = 0

# order columns nicely
df_counts = df_counts[
    ['response_set', 'question'] + list(range(1, 7))
]

# create totals
df_counts['Total'] = df_counts.loc[:, 1:6].sum(axis=1)

# vertical
'''
totals_per_question = (
    df_counts
        .groupby('question')[list(range(1, 7)) + ['Total']]
        .sum()
        .assign(response_set='Total')
        .reset_index()
)


# Combine
df_counts = pd.concat([df_counts, totals_per_question], ignore_index=True)
'''

df_counts
df_counts.head(200)



value,response_set,question,1.0,2.0,3.0,4.0,5.0,6.0,Total
0,P01,Q1_1,1,0,2,3,9,1,16
1,P01,Q1_2,1,0,1,3,10,1,16
2,P01,Q2_1,2,1,2,3,7,1,16
3,P01,Q2_2,0,0,2,5,9,0,16
4,P01,Q2_3,0,0,0,5,11,0,16
...,...,...,...,...,...,...,...,...,...
175,P20,Q2_3,0,1,2,1,8,0,12
176,P20,Q2_4,1,1,1,5,4,0,12
177,P20,Q2_5,0,0,4,1,6,1,12
178,P20,Q2_6,0,1,2,2,7,0,12


In [9]:

# Check vertical counts
total_counts_vertical = df_counts['Total'].sum()

print(f'Total responses across all questions and response sets: {total_counts_vertical}')

Total responses across all questions and response sets: 2277


In [10]:
# Tabulate by group

icai = [f'P0{i}' for i in range(1, 10)] + ['P10']   # P1–P10
ours = [f'P{i}' for i in range(11, 21)]  # P11–P20

# map
group_map = {p: 'icai' for p in icai}
group_map.update({p: 'ours' for p in ours})

df_grouped = df_counts.copy()
df_grouped['group'] = df_grouped['response_set'].map(group_map)

# safety
df_grouped['group'].value_counts(dropna=False)

# combine
df_combined = (
    df_grouped
        .groupby(['group', 'question'])[list(range(1, 7))]
        .sum()
        .reset_index()
)

df_combined['Total'] = df_combined.loc[:, 1:6].sum(axis=1)

df_combined.head(100)


value,group,question,1.0,2.0,3.0,4.0,5.0,6.0,Total
0,icai,Q1_1,1,4,8,25,86,2,126
1,icai,Q1_2,2,5,9,31,77,2,126
2,icai,Q2_1,9,6,21,38,50,2,126
3,icai,Q2_2,3,2,13,40,67,1,126
4,icai,Q2_3,2,3,7,44,69,1,126
5,icai,Q2_4,4,15,10,22,74,1,126
6,icai,Q2_5,3,5,10,44,62,2,126
7,icai,Q2_6,1,4,10,42,67,2,126
8,icai,Q2_7,4,8,18,34,57,5,126
9,ours,Q1_1,1,4,8,43,71,0,127


In [12]:
# drop col 6 (na) and recompute totals
df_combined_clean = df_combined.drop(6, axis=1)
df_combined_clean['Total'] = df_combined.loc[:, 1:5].sum(axis=1)

# sanity check: check mean and median
value_cols = [1, 2, 3, 4, 5]
values = np.array(value_cols)

# mean
df_combined_clean['Mean'] = (
    df_combined_clean[value_cols].mul(values).sum(axis=1)
    / df_combined_clean[value_cols].sum(axis=1)
)

# median
def weighted_median(row):
    freqs = row[value_cols].values
    cum_freq = np.cumsum(freqs)
    cutoff = freqs.sum() / 2
    return values[cum_freq >= cutoff][0]

# df_combined_clean['Median'] = df_combined_clean.apply(weighted_median, axis=1)

df_combined_clean

value,group,question,1.0,2.0,3.0,4.0,5.0,Total,Mean
0,icai,Q1_1,1,4,8,25,86,124,4.540323
1,icai,Q1_2,2,5,9,31,77,124,4.419355
2,icai,Q2_1,9,6,21,38,50,124,3.919355
3,icai,Q2_2,3,2,13,40,67,125,4.328000
4,icai,Q2_3,2,3,7,44,69,125,4.400000
5,icai,Q2_4,4,15,10,22,74,125,4.176000
6,icai,Q2_5,3,5,10,44,62,124,4.266129
7,icai,Q2_6,1,4,10,42,67,124,4.370968
8,icai,Q2_7,4,8,18,34,57,121,4.090909
9,ours,Q1_1,1,4,8,43,71,127,4.409449


In [13]:
total_counts_vertical_grouped = df_grouped['Total'].sum()

print(f'Total responses across all questions and response sets: {total_counts_vertical_grouped}')

Total responses across all questions and response sets: 2277


In [14]:
# quick check
icai = [f'P0{i}' for i in range(1, 10)] + ['P10']   # P1–P10
ours = [f'P{i}' for i in range(11, 21)]  # P11–P20

question_cols = [f'Q1_1', 'Q1_2'] + [f'Q2_{i}' for i in range(1, 8)]

# function to aggregate value counts for a group
def aggregate_counts(df, group, group_name):
    print(f"Aggregated value counts for {group_name}:")
    group_df = df[df['response_set'].isin(group)]
    for q in question_cols:
        print(f"{q}:")
        print(group_df[q].value_counts(dropna=False))
        print("-" * 40)
    print("=" * 60)

aggregate_counts(transformed_df, icai, "P1–P10")
aggregate_counts(transformed_df, ours, "P11–P20")

Aggregated value counts for P1–P10:
Q1_1:
Q1_1
5.0    86
4.0    25
3.0     8
2.0     4
NaN     2
6.0     2
1.0     1
Name: count, dtype: int64
----------------------------------------
Q1_2:
Q1_2
5.0    77
4.0    31
3.0     9
2.0     5
NaN     2
1.0     2
6.0     2
Name: count, dtype: int64
----------------------------------------
Q2_1:
Q2_1
5.0    50
4.0    38
3.0    21
1.0     9
2.0     6
NaN     2
6.0     2
Name: count, dtype: int64
----------------------------------------
Q2_2:
Q2_2
5.0    67
4.0    40
3.0    13
1.0     3
2.0     2
NaN     2
6.0     1
Name: count, dtype: int64
----------------------------------------
Q2_3:
Q2_3
5.0    69
4.0    44
3.0     7
2.0     3
NaN     2
1.0     2
6.0     1
Name: count, dtype: int64
----------------------------------------
Q2_4:
Q2_4
5.0    74
4.0    22
2.0    15
3.0    10
1.0     4
NaN     2
6.0     1
Name: count, dtype: int64
----------------------------------------
Q2_5:
Q2_5
5.0    62
4.0    44
3.0    10
2.0     5
1.0     3
NaN     2
6.0  

In [15]:
# MW-U test

value_cols = [1, 2, 3, 4, 5]
values = np.array(value_cols)

rows = []

for q in df_combined_clean['question'].unique():

    icai_freqs = df_combined_clean.loc[
        (df_combined_clean.question == q) &
        (df_combined_clean.group == 'icai'),
        value_cols
    ].values.flatten()

    ours_freqs = df_combined_clean.loc[
        (df_combined_clean.question == q) &
        (df_combined_clean.group == 'ours'),
        value_cols
    ].values.flatten()

    # skip questions without both groups
    if icai_freqs.sum() == 0 or ours_freqs.sum() == 0:
        continue

    icai_vals = np.repeat(values, icai_freqs.astype(int))
    ours_vals = np.repeat(values, ours_freqs.astype(int))

    u_stat, p_value = mannwhitneyu(
        icai_vals,
        ours_vals,
        alternative='two-sided'
    )

    rows.append({
        'question': q,
        'median_icai': np.median(icai_vals),
        'median_ours': np.median(ours_vals),
        'n_icai': len(icai_vals),
        'n_ours': len(ours_vals),
        'u_stat': u_stat,
        'p_value': p_value
    })

mw_table = pd.DataFrame(rows)
mw_table.head(100)


,question,median_icai,median_ours,n_icai,n_ours,u_stat,p_value
0,Q1_1,5.0,5.0,124,127,8816.0,0.056197
1,Q1_2,5.0,5.0,124,127,8491.5,0.225335
2,Q2_1,4.0,4.0,124,127,8255.0,0.487356
3,Q2_2,5.0,4.0,125,127,9202.0,0.018272
4,Q2_3,5.0,5.0,125,126,8099.0,0.663626
5,Q2_4,5.0,4.0,125,127,8922.5,0.064090
6,Q2_5,4.5,4.0,124,125,8126.5,0.471106
7,Q2_6,5.0,4.0,124,127,8359.0,0.352338
8,Q2_7,4.0,4.0,121,124,8232.0,0.163457
